# Text pre-processing

### Loading the file containing all the tracks.

In [14]:
import json

with open('dataset/tracks_lyrics.json', 'r', encoding="utf-8") as json_file:
    artists = json.load(json_file)
    
print(artists[0]["artist_id"], "\n", 
      artists[0]["artist_name"], "\n", 
      artists[0]['tracks'][0]['title'], "\n", 
      artists[0]['tracks'][0]['lyrics'][:100].lstrip(), "...", 
      sep="")

582KhTHEVOONNQLmQ5612r
Calcutta 
Tutti
Ho messo le scarpe nuove per i giorni di fango
Forse i leghisti lì in riva al Po non hanno più un ca...


### Importing lingua and removing non italian text

As the first step in our preprocessing pipeline, we utilize the Python library lingua to filter out any non-Italian text from the lyrics. Additionally, we remove any song lyrics containing fewer than 8 lines, as a standard paragraph in Italian typically consists of about 12 lines.

The lingua Python library is designed for high-accuracy language detection on short texts. By supplying a predefined set of candidate languages, lingua calculates the probability that a given text belongs to each candidate and selects the most likely match. In our preprocessing pipeline, we evaluate the text line by line; any line where Italian is not identified as the winning language is immediately removed.

In [15]:
from lingua import Language, LanguageDetectorBuilder

# Define the expected languages to give the detector context.
# This is crucial to allow the algorithm to discard non-Italian lines.
expected_languages = [
    Language.ITALIAN, 
    Language.ENGLISH, 
    Language.SPANISH, 
    Language.FRENCH, 
    Language.RUSSIAN, 
    Language.GERMAN,
    Language.PORTUGUESE,
]
detector = LanguageDetectorBuilder.from_languages(*expected_languages).build()

def filter_foreign_lines(lyrics):
    """
    Splits the lyrics into lines, detects the language of each line,
    and retains only the lines identified as Italian.
    """
    italian_lines = []
    
    # Split the lyrics by newline character
    lines = lyrics.split('\n')
    
    for line in lines:
        stripped_line = line.strip()
        
        if not stripped_line:
            continue
            
        # Detect the language of the current line
        detected_language = detector.detect_language_of(stripped_line)
        
        if detected_language == Language.ITALIAN:
            italian_lines.append(stripped_line)
        else:
            # print(f"Removed line: {stripped_line} (Language: {detected_language})")
            pass
            
    return '\n'.join(italian_lines)


# Filter tracks by language first, then by valid length
removed_tracks_count = 0
valid_tracks_count = 0
for artist in artists:
    valid_tracks = [] # List to store tracks that survive both filters
    
    for track in artist.get('tracks', []):
        raw_lyrics = track.get('lyrics', '')
        
        # Language Filter
        # Extract only the Italian lines from the lyrics
        filtered_lyrics = filter_foreign_lines(raw_lyrics)
        
        # Length Filter (Applied to the remaining Italian text)
        # Split the filtered text by newline and count the valid lines
        # We use .strip() to ignore completely empty lines or lines with just spaces
        valid_italian_lines_count = len([line for line in filtered_lyrics.split('\n') if line.strip()])
        
        # If the surviving Italian text has 8 or more lines, we keep the track
        if valid_italian_lines_count >= 8:
            track['lyrics'] = filtered_lyrics
            valid_tracks.append(track)
            valid_tracks_count += 1
        else:
            # The track is either too short overall, or too much foreign text was removed
            removed_tracks_count += 1
            
    artist['tracks'] = valid_tracks
    
print(f"Total tracks removed due to insufficient Italian content: {removed_tracks_count}")
print(f"Total valid tracks remaining after filtering: {valid_tracks_count}")

Total tracks removed due to insufficient Italian content: 6571
Total valid tracks remaining after filtering: 20616


## Tokenization, normalization, lemmetization

### spaCy

First, we cleaned the text by replacing all newline characters with spaces and normalizing it using regular expressions. We preserved the accents, as they are essential for distinguishing between different words in the Italian language. For tokenization and lemmatization, we initially turned to spaCy, a library widely regarded as the industry standard. Given the choice between their small and large Italian language packages, we opted for the large model. _(to install it: python -m spacy download it_core_news_lg)_

Because processing the lyrics sequentially proved to be too slow, we optimized our pipeline using multiprocessing.
However, upon evaluating the output on our first track **('Tutti' by Calcutta)**, we discovered significant inaccuracies in the lemmatization process. For instance, the conjugated verb **'vesto'** was incorrectly lemmatized to **'vestare'** a non-existent word in Italian. Due to these critical errors, we abandoned spaCy in favor of another library called Stanza, which is detailed below.

**WARNING** 

Using multicore processing (n_process = -1) speeds up execution but may cause conflicts with downstream libraries. If you experience issues, set n_process = 1 to run on a single core safely

In [ ]:
import spacy
import re

# Load the Italian NLP model
nlp = spacy.load("it_core_news_lg")

def regex_clean(text):
    """
    Applies extremely fast regular expression normalizations.
    It is much more efficient to do this BEFORE passing the text to spaCy.
    """
    text = text.replace('\n', ' ')
    text = re.sub(r'[^a-zA-Zàèéìíòóùú]', ' ', text).lower()
    return re.sub(r'\s+', ' ', text).strip()


# PREPARATION PHASE
# Flatten the nested dictionary structure to process lyrics in bulk
flat_tracks = []
raw_texts = []

# Collect all valid texts and their corresponding track objects
for artist in artists:
    for track in artist.get('tracks', []):
        flat_tracks.append(track)
        # Clean with Regex immediately during extraction
        raw_texts.append(regex_clean(track.get('lyrics', '')))

# --- PARALLEL PROCESSING PHASE ---
print(f"Initiating parallel processing for {len(raw_texts)} tracks...")

corpus = []

# nlp.pipe():
# - as_tuples=False: We only pass the texts
# - n_process=-1: Tells spaCy to use ALL available CPU cores. 
# - batch_size=100: Sends 100 songs to each core at a time, minimizing memory overhead
doc_generator = nlp.pipe(
    raw_texts, 
    n_process=-1, 
    batch_size=100, 
)

# Zip binds the processed 'doc' back to its original 'track' dictionary
for track, doc in zip(flat_tracks, doc_generator):
    lemmas = []
    for token in doc:
        # Filter 1: Exclude extremely short tokens
        if len(token.text) >= 2:
            lemmas.append(token.lemma_)
            
    # Reconstruct the string
    clean_lyrics = " ".join(lemmas)
    
    # Save back to the original nested dictionary structure
    track['lemmatized_lyrics'] = clean_lyrics
    
    # Add to our flat corpus list for Scikit-Learn Vectorization later
    corpus.append(clean_lyrics)

print(f"Parallel preprocessing completed")

Initiating parallel processing for 20616 tracks...
Parallel preprocessing completed


In [17]:
def highlight(text, word):

    match = re.search(fr'((?:\S+\s+){{0,3}})\b({word})\b((?:\s+\S+){{0,3}})', text, re.IGNORECASE)
    
    p_before, target, p_after = match.groups(default="")
    return f"{p_before}\033[91m{target}\033[0m{p_after}".strip().replace('\n', ' ')

print("Original Lyrics:\n", highlight(artists[0]['tracks'][0]['lyrics'], "vesto"))
print("\nLemmatized Lyrics:\n", highlight(artists[0]['tracks'][0]['lemmatized_lyrics'], "vestare"))

Original Lyrics:
 ed io mi vesto di bianco Vorrei

Lemmatized Lyrics:
 e io mi vestare di bianco volere


### NLP Pipeline with Stanza (Stanford NLP)

**Stanza** is a powerful NLP library developed by Stanford University that relies on deep neural networks. It provides excellent native support for the Italian language and is explicitly designed to leverage GPU acceleration via PyTorch.

Stanza operates on a **pipeline architecture**. A neural network cannot extract a word's root directly from raw text in a single pass; the process must be executed sequentially. Each tool in the pipeline prepares the data structure for the subsequent one. This is why we initialize our model with these three specific components: `stanza.Pipeline(lang='it', processors='tokenize,pos,lemma', use_gpu=True)`.

Before feeding the text into Stanza, we applied a text normalization step, specifically ensuring that Italian accents were preserved to maintain full semantic integrity.

Here is a breakdown of our pipeline processors:

* **`tokenize` (Tokenization):** This step takes the raw text and slices it. It first performs *Sentence Segmentation* (dividing the text into distinct sentences) and then *Tokenization* (splitting each sentence into individual words and punctuation marks).
* **`pos` (Part-Of-Speech Tagging):** It examines each token within the context of the entire sentence and assigns a grammatical label (e.g., Noun, Verb, Adjective, Adverb, Pronoun). This contextual understanding drastically improves accuracy compared to standard spaCy models. Furthermore, having defined the grammatical class of each element, we can implement an advanced filtering step: discarding structural "stop-words" (like articles and prepositions) that add no value to our upcoming Bag-of-Words (BoW) and TF-IDF matrices.
* **`lemma` (Lemmatization):** It takes the original token alongside its POS tag to accurately compute the dictionary root of the word (the lemma). Extracting the pure lemmas is our ultimate goal for cleaning the BoW and TF-IDF matrices. Thanks to the foundational work done by the POS tagger, the lemmatizer resolves contextual ambiguities and operates flawlessly.

#### Performance Note
It is important to highlight that, despite utilizing GPU acceleration, Stanza's deep learning approach is significantly slower than spaCy. On our specific hardware configuration (**CPU: Intel Ultra 7 265k, RAM: 32GB DDR5 6400MHz, GPU: Nvidia RTX 5070FE**), the entire text processing phase took approximately **3 hours** to complete.

**Bibliographic reference (Stanza):**
> Qi, P., Zhang, Y., Zhang, Y., Bolton, J., & Manning, C. D. (2020). **Stanza: A Python Natural Language Processing Toolkit for Many Human Languages**. In *Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics: System Demonstrations* (pp. 101-108).

In [ ]:
import stanza
import re

# INITIALIZATION
# Download the Italian neural model (only needs to be run once)
# stanza.download('it')

# Initialize the pipeline
# use_gpu=True automatically sends the tensors to your NVIDIA card
# processors='tokenize,pos,lemma' loads ONLY the tools we need to save RAM
nlp_stanza = stanza.Pipeline(lang='it', processors='tokenize,pos,lemma', use_gpu=True)

def regex_clean(text):
    """
    Applies fast regular expression normalizations before NLP.
    """
    text = text.replace('\n', ' ')
    text = re.sub(r'[^a-zA-Zàèéìíòóùú]', ' ', text).lower()
    return re.sub(r'\s+', ' ', text).strip()


# PREPARATION PHASE
flat_tracks = []
raw_texts = []

for artist in artists:
    for track in artist.get('tracks', []):
        flat_tracks.append(track)
        raw_texts.append(regex_clean(track.get('lyrics', '')))


# GPU NEURAL PROCESSING PHASE
print(f"Initiating Stanza Neural processing for {len(raw_texts)} tracks...")

corpus = []

# Define the set of valid POS tags ONCE
VALID_POS_TAGS = {'NOUN', 'VERB', 'ADJ', 'ADV'}

# Process each track. Stanza automatically handles GPU batching internally
# for document objects, though it processes them sequentially in the loop.
for track, text in zip(flat_tracks, raw_texts):
    
    # Pass the text to the neural network
    doc = nlp_stanza(text)
    
    lemmas = []
    
    # Stanza structures text into sentences, then words
    for sentence in doc.sentences:
        for word in sentence.words:
            
            # Filter 1: Check the Part-Of-Speech (POS) tag against our pre-defined set. 
            # We ONLY keep Nouns (NOUN), Verbs (VERB), Adjectives (ADJ), and Adverbs (ADV)
            # This automatically acts as an incredibly powerful, grammar-based stop-word remover
            if word.upos in VALID_POS_TAGS:
                # Filter 2: extremely short tokens
                if len(word.text) >= 2:
                    # Stanza stores the base form in word.lemma
                    lemmas.append(word.lemma)
                
    clean_lyrics = " ".join(lemmas)
    track['lemmatized_lyrics'] = clean_lyrics
    corpus.append(clean_lyrics)

print(f"Neural processing completed.")

2026-06-10 00:16:31 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-06-10 00:16:31 INFO: Downloaded file to /home/gabriele11231/.cache/stanza/1.12.0/resources/resources.json
2026-06-10 00:16:31 WARNING: Language it package default expects mwt, which has been added
2026-06-10 00:16:32 INFO: Loading these models for language: it (Italian):
| Processor | Package           |
---------------------------------
| tokenize  | combined_nocharlm |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2026-06-10 00:16:32 INFO: Using device: cuda
2026-06-10 00:16:32 INFO: Loading: tokenize
2026-06-10 00:16:32 INFO: Loading: mwt
2026-06-10 00:16:32 INFO: Loading: pos
2026-06-10 00:16:33 INFO: Loading: lemma
2026-06-10 00:16:33 INFO: Done loading processors!


Initiating Stanza Neural processing for 20616 tracks...
Neural processing completed.


As we can see below, using againg the song 'Tutti' by Calcutta as an example, the word vesto is now lemmatized as vestire.

In [4]:
def highlight(text, word):

    match = re.search(fr'((?:\S+\s+){{0,3}})\b({word})\b((?:\s+\S+){{0,3}})', text, re.IGNORECASE)
    
    p_before, target, p_after = match.groups(default="")
    return f"{p_before}\033[91m{target}\033[0m{p_after}".strip().replace('\n', ' ')

print("Original Lyrics:\n", highlight(artists[0]['tracks'][0]['lyrics'], "vesto"))
print("\nLemmatized Lyrics:\n", highlight(artists[0]['tracks'][0]['lemmatized_lyrics'], "vestire"))

Original Lyrics:
 ed io mi vesto di bianco Vorrei

Lemmatized Lyrics:
 più capobranco rivoluzione vestire bianco tenere mano


Since Stanza is computationally expensive and takes a long time to run, we save its output for future use.

In [ ]:
with open("dataset/tracks_lyrics_preprocessed.json", 'w', encoding="utf-8") as json_file:
        json.dump(artists, json_file, ensure_ascii=False, indent=4)